# Binance +300% 因子挖掘

定义：在过去30天窗口内部，遍历任意较早4小时收盘价/最低价到任意后续4小时最高价；+300% 表示价格达到4倍，不是首日到末日收益。预测标签使用次日开盘后14日内最高价达到4倍。

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPORT_DIR = Path('F:/9_Crypto/crypto_trading_system/reports/binance_300pct_factor_mining_2026-07-18')
summary = json.loads((REPORT_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
metrics = json.loads((REPORT_DIR / 'model_metrics.json').read_text(encoding='utf-8'))
summary['label_definition']

In [ ]:
runups = pd.read_csv(REPORT_DIR / 'current_30d_runups.csv')
runups[['symbol', 'classification', 'close_to_later_high_return', 'low_to_later_high_return', 'close_entry_time', 'close_peak_time']]

In [ ]:
simple = metrics['simple_fragility_score']
pd.DataFrame([
    {'period': 'train', **simple['train']},
    {'period': 'test', **simple['test']},
])[['period', 'n', 'positives', 'base_rate', 'auc', 'top_2pct_precision', 'top_2pct_lift']]

In [ ]:
factors = pd.read_csv(REPORT_DIR / 'factor_stability.csv')
factors.loc[factors['stable'], ['factor', 'direction', 'train_auc', 'test_auc', 'train_top_decile_lift', 'test_top_decile_lift']].head(12)

In [ ]:
test_strategy = pd.read_csv(REPORT_DIR / 'strategy_backtest_summary.csv')
train_strategy = pd.read_csv(REPORT_DIR / 'strategy_train_summary.csv')
pd.concat([
    train_strategy.query("top_k == 3").assign(period='train'),
    test_strategy.query("strategy == 'simple_fragility_score' and top_k == 3").assign(period='test'),
])[['period', 'median_phase_mean_net_return', 'median_phase_median_net_return', 'median_phase_win_rate', 'median_phase_hit_target_rate', 'worst_trade_drawdown']]

完整刷新可运行 `run_300pct_factor_mining.py`。该脚本设置 +300% 阈值并调用通用全市场管线。